# NumPy Broadcasting — Runnable Examples

Examples demonstrate shape compatibility, row and column broadcasting, normalization, memory behavior, and applied data preparation.

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.3.5


## 1. Broadcast two arrays

In [2]:
column = np.array([[1], [2], [3]])
row = np.array([[10, 20, 30, 40]])
result = column + row
print(result)
print("input shapes:", column.shape, row.shape)
print("result shape:", result.shape)

[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]
input shapes: (3, 1) (1, 4)
result shape: (3, 4)


## 2. Broadcasting shape predictions

In [3]:
examples = [
    ((3, 4), (4,)),
    ((3, 4), (3, 1)),
    ((3, 1), (1, 4)),
    ((2, 3, 4), (3, 1)),
    ((5, 1, 4), (1, 3, 4)),
]
for shape_a, shape_b in examples:
    print(shape_a, "+", shape_b, "->", np.broadcast_shapes(shape_a, shape_b))

(3, 4) + (4,) -> (3, 4)
(3, 4) + (3, 1) -> (3, 4)
(3, 1) + (1, 4) -> (3, 4)
(2, 3, 4) + (3, 1) -> (2, 3, 4)
(5, 1, 4) + (1, 3, 4) -> (5, 3, 4)


## 3. Per-column and per-row operations

In [4]:
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
column_offsets = np.array([100, 200, 300])
row_offsets = np.array([1, 2, 3, 4])
print("per-column offsets:\n", matrix + column_offsets)
print("per-row offsets:\n", matrix + row_offsets[:, None])

per-column offsets:
 [[101 202 303]
 [104 205 306]
 [107 208 309]
 [110 211 312]]
per-row offsets:
 [[ 2  3  4]
 [ 6  7  8]
 [10 11 12]
 [14 15 16]]


## 4. Comparisons and `np.where` with broadcasting

In [5]:
scores = np.array([[65, 72, 80], [90, 55, 68], [75, 88, 92]])
thresholds = np.array([70, 70, 75])
print("pass mask:\n", scores >= thresholds)
print("replace below threshold with zero:\n", np.where(scores < thresholds, 0, scores))

pass mask:
 [[False  True  True]
 [ True False False]
 [ True  True  True]]
replace below threshold with zero:
 [[ 0 72 80]
 [90  0  0]
 [75 88 92]]


## 5. Column standardization

In [6]:
X = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
means = X.mean(axis=0)
stds = X.std(axis=0)
safe_stds = np.where(stds == 0, 1.0, stds)
Z = (X - means) / safe_stds
print("means:", means, "shape:", means.shape)
print("standardized:\n", Z)
print("column means after standardization:", Z.mean(axis=0))

means: [ 2. 20.] shape: (2,)
standardized:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
column means after standardization: [0. 0.]


## 6. Safe row-wise min-max normalization

In [7]:
def normalize_rows(X):
    X = np.asarray(X, dtype=float)
    mins = X.min(axis=1, keepdims=True)
    spans = X.max(axis=1, keepdims=True) - mins
    safe_spans = np.where(spans == 0, 1.0, spans)
    return (X - mins) / safe_spans

example = np.array([[2, 4, 6], [5, 5, 5], [10, 20, 30]], dtype=float)
print(normalize_rows(example))

[[0.  0.5 1. ]
 [0.  0.  0. ]
 [0.  0.5 1. ]]


## 7. Broadcasting views and memory

In [8]:
row = np.array([1, 2, 3])
view = np.broadcast_to(row, (4, 3))
print(view)
print("shape:", view.shape)
print("writeable:", view.flags.writeable)
print("shares memory:", np.shares_memory(row, view))
print("float64 result bytes for shape (1000, 2000):", 1000 * 2000 * np.dtype(np.float64).itemsize)

[[1 2 3]
 [1 2 3]
 [1 2 3]
 [1 2 3]]
shape: (4, 3)
writeable: False
shares memory: True
float64 result bytes for shape (1000, 2000): 16000000


## 8. Detect incompatible shapes

In [9]:
def try_broadcast_shape(shape_a, shape_b):
    try:
        return np.broadcast_shapes(shape_a, shape_b)
    except ValueError:
        return "incompatible"

print(try_broadcast_shape((3, 4), (4,)))
print(try_broadcast_shape((3, 4), (3,)))
assert try_broadcast_shape((3, 1), (1, 4)) == (3, 4)
assert try_broadcast_shape((3, 4), (3,)) == "incompatible"
print("All checks passed.")

(3, 4)
incompatible
All checks passed.
